In [ ]:
!pip install -q torch torchaudio transformers timm
!pip install -q librosa soundfile scipy scikit-learn xgboost
!pip install -q pandas numpy matplotlib seaborn tqdm
!pip install -q grad-cam

In [ ]:
import os, sys, glob, json, random, warnings, math, copy
from pathlib import Path
from collections import Counter, defaultdict

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

In [ ]:
import torchaudio
import torchaudio.transforms as T
import librosa
import scipy.signal as signal

In [ ]:
from sklearn.metrics import (
classification_report, confusion_matrix, f1_score,
precision_recall_fscore_support, roc_auc_score
)
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Torch:", torch.__version__)

Device: cuda
Torch: 2.11.0+cu128


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
PROJECT_ROOT = Path("/content/drive/MyDrive/respiratory_project")
ICBHI_ROOT = PROJECT_ROOT / "ICBHI_final_database"
SPLIT_FILE = PROJECT_ROOT / "ICBHI_challenge_train_test.txt"
DIAG_FILE = PROJECT_ROOT / "ICBHI_Challenge_diagnosis.txt"

In [ ]:
CACHE_DIR = PROJECT_ROOT / "cache"
CKPT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results"
FIG_DIR = PROJECT_ROOT / "figures"

In [ ]:
for d in [CACHE_DIR, CKPT_DIR, RESULTS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
##adding hyper-parameters for audio(initiaslising)

In [ ]:
CFG = {
    "sample_rate": 4000,
    "bandpass_low": 50,
    "bandpass_high": 1800,
    "cycle_seconds": 6.0,
    "n_mels": 64,
    "n_fft": 512,
    "hop_length": 160,
    "win_length": 400,
    "batch_size": 32,
    "epochs": 60,
    "lr_head": 1e-3,
    "lr_backbone": 1e-4,
    "weight_decay": 1e-4,
    "patience": 12,
    "num_workers": 2,
    "n_seeds": 3,
    "task": "adventitious_4cls", "label_smoothing": 0.05,
    "mixup_alpha": 0.2,
}
print(CFG)

{'sample_rate': 4000, 'bandpass_low': 50, 'bandpass_high': 1800, 'cycle_seconds': 6.0, 'n_mels': 64, 'n_fft': 512, 'hop_length': 160, 'win_length': 400, 'batch_size': 32, 'epochs': 60, 'lr_head': 0.001, 'lr_backbone': 0.0001, 'weight_decay': 0.0001, 'patience': 12, 'num_workers': 2, 'n_seeds': 3, 'task': 'adventitious_4cls', 'label_smoothing': 0.05, 'mixup_alpha': 0.2}


In [ ]:
#Parsing filenames,diagnosis,splits, and cycle annotations into one DataFrame.

In [ ]:
def parse_diagnosis(diag_path):
    df = pd.read_csv(diag_path, sep="\t", header=None, names=["patient_id", "diagnosis"])
    df["patient_id"] = df["patient_id"].astype(str)
    return df

In [ ]:
#converted patiernt id to string\

In [ ]:
def parse_split(split_path):
    df = pd.read_csv(split_path, sep="\t", header=None, names=["recording_id", "split"])
    return df

In [ ]:
def parse_filename(fname):
    parts = Path(fname).stem.split("_")
    return {
    "recording_id": Path(fname).stem,
    "patient_id": parts[0],
    "rec_index": parts[1],
    "chest_loc": parts[2],
    "acq_mode": parts[3],
    "device": parts[4],
    }

In [ ]:
def parse_cycles(txt_path):
    df = pd.read_csv(txt_path, sep="\t", header=None,
    names=["start", "end", "crackles", "wheezes"])
    return df

In [ ]:
wavs = sorted(glob.glob(str(ICBHI_ROOT / "*.wav")))
print("WAV files:", len(wavs))

WAV files: 920


In [ ]:
diag_df = parse_diagnosis(DIAG_FILE)
split_df = parse_split(SPLIT_FILE)

In [ ]:
records = []
for wav in wavs:
    meta = parse_filename(wav)
    txt = wav.replace(".wav", ".txt")
    if not os.path.exists(txt):
        continue
    cycles = parse_cycles(txt)
    for i, row in cycles.iterrows():
        records.append({
            **meta, "wav_path": wav, "cycle_idx": i,
            "start": float(row["start"]), "end": float(row["end"]),
            "crackles": int(row["crackles"]), "wheezes": int(row["wheezes"]),
    })

In [ ]:
cycles_df = pd.DataFrame(records)
cycles_df = cycles_df.merge(diag_df, on="patient_id", how="left")
cycles_df = cycles_df.merge(split_df, on="recording_id", how="left")
print("Total cycles:", len(cycles_df))
print(cycles_df["diagnosis"].value_counts())
print(cycles_df["split"].value_counts())

Total cycles: 6898
diagnosis
COPD              5746
Healthy            322
Pneumonia          285
URTI               243
Bronchiolitis      160
Bronchiectasis     104
LRTI                32
Asthma               6
Name: count, dtype: int64
split
train    4131
test     2756
Name: count, dtype: int64


In [ ]:
def make_disease_label(row):
        d = row["diagnosis"]
        if d == "Healthy": return 0
        if d == "COPD": return 1
        if d == "Pneumonia": return 2
        if d in ["Asthma","Bronchiectasis","Bronchiolitis","URTI","LRTI"]:
            return 3
        return -1

In [ ]:
def make_adventitious_label(row):
    c, w = row["crackles"], row["wheezes"]

    if c == 0 and w == 0:
        return 0      # Normal
    if c == 1 and w == 0:
        return 1      # Crackle
    if c == 0 and w == 1:
        return 2      # Wheeze
    if c == 1 and w == 1:
        return 3      # Both

    return -1


if CFG["task"] == "adventitious_4cls":
    cycles_df["label"] = cycles_df.apply(make_adventitious_label, axis=1)
    LABEL_NAMES = ["Normal", "Crackle", "Wheeze", "Both"]
else:
    cycles_df["label"] = cycles_df.apply(make_disease_label, axis=1)
    LABEL_NAMES = ["Healthy", "COPD", "Pneumonia", "OtherAbnormal"]

In [ ]:
cycles_df = cycles_df[cycles_df["label"] >= 0].reset_index(drop=True)

NUM_CLASSES = len(LABEL_NAMES)

print("Task:", CFG["task"], "| Classes:", LABEL_NAMES)
print(cycles_df["label"].value_counts().sort_index())

Task: adventitious_4cls | Classes: ['Normal', 'Crackle', 'Wheeze', 'Both']
label
0    3642
1    1864
2     886
3     506
Name: count, dtype: int64


In [ ]:
print("Task:", CFG["task"], "| Classes:", LABEL_NAMES)
print(cycles_df["label"].value_counts().sort_index())
print(cycles_df["label"].head())


Task: adventitious_4cls | Classes: ['Normal', 'Crackle', 'Wheeze', 'Both']
label
0    3642
1    1864
2     886
3     506
Name: count, dtype: int64
0    0
1    0
2    0
3    0
4    0
Name: label, dtype: int64


In [ ]:
cycles_df = cycles_df[cycles_df["label"] >= 0].reset_index(drop=True)
NUM_CLASSES = len(LABEL_NAMES)
print("Task:", CFG["task"], "| Classes:", LABEL_NAMES)
print(cycles_df["label"].value_counts().sort_index())

Task: adventitious_4cls | Classes: ['Normal', 'Crackle', 'Wheeze', 'Both']
label
0    3642
1    1864
2     886
3     506
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(cycles_df, groups=cycles_df["patient_id"])
)

train_df = cycles_df.iloc[train_idx].reset_index(drop=True)
test_df = cycles_df.iloc[test_idx].reset_index(drop=True)

In [ ]:
set(train_df["patient_id"]) & set(test_df["patient_id"])

set()

In [ ]:
official_overlap = set(train_df["patient_id"]) & set(test_df["patient_id"])
assert len(official_overlap) == 0, (
    f"LEAK in official split! shared patients: {sorted(official_overlap)}")
print("Official split patient-disjoint: OK")

Official split patient-disjoint: OK


In [ ]:
from asyncio import transports
train_patients = train_df["patient_id"].unique()
rng = np.random.RandomState(SEED)
val_patients = set(rng.choice(train_patients,size=int(0.15*len(train_patients)), replace=False))
val_df = train_df[train_df["patient_id"].isin(val_patients)].reset_index(drop=True)
train_df = train_df[~train_df["patient_id"].isin(val_patients)].reset_index(drop=True)

In [ ]:
assert len(set(train_df["patient_id"]) & set(val_df["patient_id"])) == 0
assert len(set(train_df["patient_id"]) & set(test_df["patient_id"])) == 0
print("Train:", len(train_df), "| Val:", len(val_df), "| Test:", len(test_df))

Train: 4191 | Val: 1004 | Test: 1703


In [ ]:
from scipy.signal import butter, sosfiltfilt

In [ ]:

def make_bandpass(sr, low, high, order=4):
    return butter(order, [low, high], btype="band", fs=sr, output="sos")

In [ ]:
SOS = make_bandpass(CFG["sample_rate"], CFG["bandpass_low"], CFG["bandpass_high"])

In [ ]:

def load_cycle(wav_path, start, end, target_sr):
    y, sr = librosa.load(wav_path, sr=target_sr, mono=True,
                        offset=start, duration=max(0.0, end-start))
    if len(y) == 0:
        return np.zeros(int(target_sr * 0.1), dtype=np.float32)
    return y.astype(np.float32)

In [ ]:

def preprocess(y):
    target_len = int(CFG["sample_rate"] * CFG["cycle_seconds"])
    min_len = 3 * (4 * 2) # order=4 -> safe minimum length
    if len(y) < max(16, min_len):
        return np.zeros(target_len, dtype=np.float32)
    y = sosfiltfilt(SOS, y).astype(np.float32)
    rms = np.sqrt(np.mean(y**2) + 1e-9)
    y = y / (rms + 1e-9) * 0.1
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]
    return y

In [ ]:
mel_extractor = T.MelSpectrogram(
    sample_rate=CFG["sample_rate"], n_fft=CFG["n_fft"],
    win_length=CFG["win_length"], hop_length=CFG["hop_length"],
    n_mels=CFG["n_mels"], f_min=CFG["bandpass_low"],
    f_max=CFG["bandpass_high"], power=2.0,
)

NameError: name 'T' is not defined

In [ ]:
amp_to_db = T.AmplitudeToDB(top_db=80)

In [ ]:
def wav_to_logmel(y):
    t = torch.from_numpy(y).float().unsqueeze(0)
    return amp_to_db(mel_extractor(t)).squeeze(0).numpy()

In [ ]:
def cache_features(df, name):
    cache_path = CACHE_DIR / f"{name}_{CFG['task']}_logmel.npz"
    if cache_path.exists():
        print(f"Loading cached {name}")
        d = np.load(cache_path, allow_pickle=True)
        return d["X"], d["y"], d["pid"], d["dev"]
    Xs, ys, pids, devs = [], [], [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Caching {name}"):
        y = load_cycle(row["wav_path"], row["start"], row["end"], CFG["sample_rate"])
        y = preprocess(y)
        Xs.append(wav_to_logmel(y).astype(np.float32))
        ys.append(int(row["label"])); pids.append(str(row["patient_id"]))
        devs.append(str(row["device"]))
    X = np.stack(Xs)
    np.savez_compressed(cache_path, X=X, y=np.array(ys),
                        pid=np.array(pids), dev=np.array(devs))
    print(f"Cached {cache_path} | shape {X.shape}")
    return X, np.array(ys), np.array(pids), np.array(devs)

In [ ]:
X_train, y_train, pid_train, dev_train = cache_features(train_df, "train")
X_val, y_val, pid_val, dev_val = cache_features(val_df, "val")
X_test, y_test, pid_test, dev_test = cache_features(test_df, "test")
print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

In [1]:
from collections import Counter

In [ ]:
cnt = Counter(y_train.tolist()); n = sum(cnt.values())
class_weights = torch.tensor(
    [n / (NUM_CLASSES * cnt[c]) for c in range(NUM_CLASSES)],
    dtype=torch.float32).to(DEVICE)
print("Class weights:", class_weights.cpu().numpy())